In [1]:
import torch
import torchaudio
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from transformers import HubertModel
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


c:\Users\Asus\anaconda3\envs\wav2vec_ser\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda


In [2]:
df_train = pd.read_csv(r"D:\SER_Cross\data\processed\train.csv")
df_val   = pd.read_csv(r"D:\SER_Cross\data\processed\val.csv")
df_test  = pd.read_csv(r"D:\SER_Cross\data\processed\test.csv")

df_all = pd.concat([df_train, df_val, df_test], ignore_index=True)

emotion_map = {
    "neutral": "neutral_calm",
    "calm": "neutral_calm",
    "happy": "happy",
    "sad": "sad",
    "angry": "angry"
}

df_all["emotion_4"] = df_all["emotion"].map(emotion_map)
df_all = df_all.dropna(subset=["emotion_4"])


In [3]:
le = LabelEncoder()
y = le.fit_transform(df_all["emotion_4"])
print(le.classes_)


['angry' 'happy' 'neutral_calm' 'sad']


In [4]:
SAMPLE_RATE = 16000
MAX_LEN = 3 * SAMPLE_RATE   # 🔥 3 seconds (critical for speed)

class SERDataset(Dataset):
    def __init__(self, df, labels):
        self.df = df.reset_index(drop=True)
        self.labels = labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        path = self.df.loc[idx, "path"]
        wav, sr = torchaudio.load(path)
        wav = wav.mean(0)

        if sr != SAMPLE_RATE:
            wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)

        if wav.shape[0] < MAX_LEN:
            wav = torch.nn.functional.pad(wav, (0, MAX_LEN - wav.shape[0]))
        else:
            wav = wav[:MAX_LEN]

        return wav, self.labels[idx]


In [5]:
from sklearn.model_selection import train_test_split

X_train_df, X_test_df, y_train, y_test = train_test_split(
    df_all, y, test_size=0.25, stratify=y, random_state=42
)

train_ds = SERDataset(X_train_df, y_train)
test_ds  = SERDataset(X_test_df, y_test)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=4, shuffle=False)


In [6]:
hubert = HubertModel.from_pretrained("facebook/hubert-base-ls960").to(device)

for p in hubert.parameters():
    p.requires_grad = False

print("HuBERT loaded & frozen")


c:\Users\Asus\anaconda3\envs\wav2vec_ser\lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
c:\Users\Asus\anaconda3\envs\wav2vec_ser\lib\site-packages\torch\_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of the model checkpoint at facebook/hubert-base-ls960 were not used when initializing HubertModel: ['encoder.pos_conv_embed.conv.weight_g', 'encoder.pos_conv_embed.conv.weight_v']
- This IS expected if you are initializing HubertModel from the checkpoint of a model trained 

HuBERT loaded & frozen


In [7]:
class HubertSER(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Linear(768, 4)

    def forward(self, wav):
        out = self.encoder(wav).last_hidden_state   # [B, T, 768]
        out = out.transpose(1, 2)
        out = self.pool(out).squeeze(-1)
        return self.classifier(out)


In [8]:
model = HubertSER(hubert).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=1e-3)


In [ ]:
from tqdm import tqdm

for epoch in range(3):
    model.train()
    correct, total = 0, 0

    for wav, label in tqdm(train_loader):
        wav = wav.to(device)
        label = label.to(device)

        optimizer.zero_grad()
        logits = model(wav)
        loss = criterion(logits, label)
        loss.backward()
        optimizer.step()

        preds = logits.argmax(1)
        correct += (preds == label).sum().item()
        total += label.size(0)

    print(f"Epoch {epoch+1} Train Acc:", correct / total)


  0%|          | 0/1081 [00:00<?, ?it/s]

In [ ]:
model.eval()
correct, total = 0, 0

with torch.no_grad():
    for wav, label in test_loader:
        wav = wav.to(device)
        label = label.to(device)

        logits = model(wav)
        preds = logits.argmax(1)
        correct += (preds == label).sum().item()
        total += label.size(0)

test_acc = correct / total
print("HuBERT 4-Class Test Accuracy:", test_acc)


In [ ]:
y_true, y_pred = [], []

with torch.no_grad():
    for wav, label in test_loader:
        wav = wav.to(device)
        logits = model(wav)
        preds = logits.argmax(1).cpu().numpy()
        y_pred.extend(preds)
        y_true.extend(label.numpy())

print(classification_report(y_true, y_pred, target_names=le.classes_))
